> Copyright 2026 Google LLC
>
> Licensed under the Apache License, Version 2.0 (the "License"); \
> you may not use this file except in compliance with the License. \
> You may obtain a copy of the License at
>
>    https://www.apache.org/licenses/LICENSE-2.0
>
> Unless required by applicable law or agreed to in writing, software \
> distributed under the License is distributed on an "AS IS" BASIS, \
> WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. \
> See the License for the specific language governing permissions and \
> limitations under the License.

# Shape Shifter
This project provides a framework for generating text and image ads using Google's Gemini Model, designed to be scalable and user-friendly for marketing teams. Please visit the [GitHub Repository](https://github.com/pkenjora/shape_shifter/tree/main/basic) for all documentation.

In [ ]:
!(ls shape_shifter || git clone https://github.com/google-marketing-solutions/shape_shifter) > /dev/null
!cd shape_shifter
!pip install uv
!echo $(ls .venv || uv venv .venv) > /dev/null
!source .venv/bin/activate
!uv sync

## Authenticate (Colab only)

If you run this in Cloud Shell Editor or VSCode, you can skip this step.

In [ ]:
from google.colab import auth
auth.authenticate_user()

And set your GCP project. Edit this to be whatever Google Cloud Project you use.

In [ ]:
# @title Set required variables
GCP_PROJECT = "seancjones-playground-360618" # @param {"type":"string"}
GCS_BUCKET = "abcd-seancjones-storage"  # @param {"type": "string"}

%env GCP_PROJECT=$GCP_PROJECT
%env GCS_BUCKET=$GCS_BUCKET

In [4]:
import os
from google.genai import types
from PIL import Image

In [5]:
from shape_shifter.services.dv360 import GCSDV360Service
from shape_shifter.services.google_ads import GCSGoogleAdsService
dv360_service = GCSDV360Service(gcs_bucket=os.environ.get('GCS_BUCKET'))
google_ads_service = GCSGoogleAdsService(gcs_bucket=os.environ.get('GCS_BUCKET'))

# Direct usage of GeminiTextGenerationService
The following cell shows how to use the `GeminiTextGenerationService` directly to generate ad copy.

In [ ]:
import importlib
import os
from services import text_generation
importlib.reload(text_generation)
GeminiTextGenerationService = text_generation.GeminiTextGenerationService
from services.text_generation_schemas import AdCopyInput 

# Make sure to set your GCP_PROJECT environment variable in a previous cell
project_id = os.environ.get('GCP_PROJECT')
location = 'us-central1' # Or any other supported location

with open('ad.jpg', 'rb') as fh:
    image_data = fh.read()

base_image = types.Blob(data=image_data, display_name='Coffee Ad', mime_type='image/png')

# Initialize the service
if project_id:
    text_generation_service = GeminiTextGenerationService(project_id=project_id, location=location, model_name='gemini-2.5-flash')
    # Create a sample prompt
    sample_prompt = AdCopyInput(
        context='Add something iconic to the target',
        product='Dark Roast Coffee',
        targets=['Coffee lovers in San Francisco']
    )
    
    # Generate ads
    generated_ads = text_generation_service.generate_ad_plan(base_image=base_image, inputs=[sample_prompt])
    
    # Print the results
    for ad in generated_ads:
        print(ad)
else:
    print('GCP_PROJECT environment variable not set. Please set it in a previous cell.')

# Direct usage of ImageEditingService
The following cell shows how to use the `ImageEditingService` to apply the generated ad copy to the base image.

In [ ]:
import io
import base64
# import importlib

from services import image_editing
# importlib.reload(image_editing)
VertexAIImageEditingService = image_editing.VertexAIImageEditingService

# This cell assumes `project_id`, `location`, `generated_ads`, and `base_image` are available from the previous cells.

if project_id and 'generated_ads' in locals() and generated_ads:
    # The edit_image method expects bytes, so we'll use the raw image data.
    with open('ad.jpg', 'rb') as f:
        image_bytes = f.read()
    
    # The service needs an API key.
    # You can get one from aistudio.google.com.
    # For this demo, we'll try to get it from the environment variables.
    
    image_editing_service = VertexAIImageEditingService(
        project_id=project_id, 
        location=location, 
    )
    
    # Use the first ad creative generated in the previous step
    ad_to_apply = generated_ads[0]
    
    # Construct the prompt for the image editing service from the ad creative's plan.
    edit_prompt = ' '.join(ad_to_apply.plan)
    
    # Generate the edited image
    edited_image_bytes = image_editing_service.edit_image(
        prompt=edit_prompt, 
        base_image_bytes=image_bytes
    )
    
    # Display the edited image
    if edited_image_bytes:
        print("Displaying Edited Image:")
        display(edited_image_bytes)
    else:
        print("Image editing resulted in no output.")
else:
    print("Required variables (project_id, generated_ads, base_image) not found. Please run the previous cells first.")
with open('output.png', 'wb') as fh:
    fh.write(edited_image_bytes.image_bytes)

In [ ]:
from IPython.display import HTML
display(HTML('<img src="output.png">'))